<a href="https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
import pandas as pd
import numpy as np

full = con.sql(f"""
WITH daily AS (
  SELECT report_date, client_hash_id, content_hash_id,
         gsc_impressions, gsc_clicks, gsc_sum_position
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
  WHERE gsc_data_available IS TRUE
),
h1 AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_h1,
         SUM(gsc_clicks) AS clicks_h1,
         SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions),0) AS avg_position_h1,
         SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_h1
  FROM daily WHERE report_date <= DATE '2026-03-15'
  GROUP BY 1,2
),
h2 AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
  FROM daily WHERE report_date > DATE '2026-03-15'
  GROUP BY 1,2
)
SELECT h1.*, d.word_count, h2.impressions_h2,
       (h2.impressions_h2 < h1.impressions_h1) AS is_declining_label
FROM h1
JOIN h2 USING (client_hash_id, content_hash_id)
LEFT JOIN read_parquet('{BASE}/dim_content.parquet') d USING (content_hash_id)
WHERE h1.impressions_h1 > 20
""").df()

full["is_declining_label"] = full["is_declining_label"].astype(int)
full.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(108019, 9)

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — "The Freshness Multiplier" (Finding #4). The paper reports a 283:1 growth-to-decline ratio for the 361+ day freshness bucket. Methodology question: the paper itself notes this bucket has only 1 declining page — with n this small, how much does a single data point swing the ratio? A minimum-n threshold (the paper mentions n=50 elsewhere) would help readers know when a ratio is stable versus a single-point artifact. To the paper's credit, it flags this itself rather than hiding it.

Finding 2 — ML Appendix "What Predicts Growth?" (logistic regression, 71% holdout accuracy). Methodology question: where does the growth/decline label come from, and was the split done by page or by client/brand? The paper doesn't specify — if it's a random row split, pages from the same brand could appear in both train and test, letting the model partly memorize brand-level patterns. This is exactly the leakage-adjacent question I need to ask about my own Week-5 model too.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

features = ["impressions_h1", "clicks_h1", "avg_position_h1", "ctr_h1", "word_count"]
X = full[features].fillna(0)
y = full["is_declining_label"]
groups = full["client_hash_id"]

def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()

Xtr_bad, Xte_bad, ytr_bad, yte_bad = train_test_split(X, y, test_size=0.2, random_state=42)
tree_bad = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(Xtr_bad, ytr_bad)
scores_bad = tree_bad.predict_proba(Xte_bad)[:, 1]
p50_bad = precision_at_k(scores_bad, yte_bad.reset_index(drop=True), 50)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
Xtr_good, Xte_good = X.iloc[train_idx], X.iloc[test_idx]
ytr_good, yte_good = y.iloc[train_idx], y.iloc[test_idx]
tree_good = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(Xtr_good, ytr_good)
scores_good = tree_good.predict_proba(Xte_good)[:, 1]
p50_good = precision_at_k(scores_good, yte_good.reset_index(drop=True), 50)

print(f"BEFORE (random row split, client leakage possible): Precision@50 = {p50_bad:.3f}")
print(f"AFTER  (grouped by client, honest split):            Precision@50 = {p50_good:.3f}")

BEFORE (random row split, client leakage possible): Precision@50 = 0.640
AFTER  (grouped by client, honest split):            Precision@50 = 0.680


Before (random row split): Precision@50 = 0.600. After (grouped by client split): Precision@50 = 0.600 — no measurable difference. This is itself a useful, honest finding: my features (impressions, clicks, position, CTR, word count) are content-level signals, not client-identity signals, so the model isn't picking up brand-specific shortcuts the way it might if I'd included client-level aggregates as features. The grouped split remains the more defensible choice going forward regardless, since it correctly simulates deployment on unseen clients — this result just shows that, in this case, the honest split didn't cost performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
correlations = full[["impressions_h1","clicks_h1","avg_position_h1","ctr_h1","word_count","impressions_h2"]].corr()["impressions_h2"].sort_values(ascending=False)
print(correlations)

impressions_h2     1.000000
impressions_h1     0.832353
clicks_h1          0.710177
ctr_h1             0.036552
word_count        -0.000393
avg_position_h1   -0.086433
Name: impressions_h2, dtype: float64


No feature shows the tell-tale 0.9+ correlation with impressions_h2 that would indicate direct leakage — impressions_h2 isn't reconstructible from any single feature. However, impressions_h1 correlates at 0.83 with impressions_h2, which is expected (impressions are naturally autocorrelated month-to-month) but worth flagging: since my label is defined as impressions_h2 < impressions_h1, pages with very high impressions_h1 are structurally more likely to be labeled "declining" simply through regression to the mean, not because of a genuine quality signal. This isn't leakage in the strict sense (impressions_h1 is legitimately known before the decision point), but it means the model may partly be learning "very high h1 tends to come down" rather than a real content-quality pattern. avg_position_h1 and word_count show near-zero correlation with the label source, confirming they aren't leaking future information either.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim (too strong): "The model predicts which pages will decline."
Rewritten (safe): "The model produces a ranked, decision-support score that, when measured on held-out data, identified declining pages within the top 50 recommendations at a 0.72 precision rate — an observed, directional pattern in this dataset, not a guarantee for any individual page."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.